In [2]:
import sys
print(sys.executable)

c:\Users\divya\Desktop\AI\LangChain_Lab\.venv\Scripts\python.exe


In [3]:
import sys
from pathlib import Path

PROJECT_ROOT = Path.cwd().parent
sys.path.insert(0, str(PROJECT_ROOT))

from common.llm import get_llm

In [4]:
from common.llm import get_llm
from langchain_core.prompts import ChatPromptTemplate
from langchain_core.output_parsers import StrOutputParser
from langchain_core.runnables import RunnableLambda, RunnableParallel, RunnableBranch

In [8]:
outline_prompt = ChatPromptTemplate.from_messages([
    ("system", "You are a content strategist. Create a short bullet-point outline (4-6 points) for the given topic."),
    ("human", "{topic}"),
])

outline_chain = outline_prompt | get_llm(0.7) | StrOutputParser()
print(type(outline_chain))
print(outline_chain)

<class 'langchain_core.runnables.base.RunnableSequence'>
first=ChatPromptTemplate(input_variables=['topic'], input_types={}, partial_variables={}, messages=[SystemMessagePromptTemplate(prompt=PromptTemplate(input_variables=[], input_types={}, partial_variables={}, template='You are a content strategist. Create a short bullet-point outline (4-6 points) for the given topic.'), additional_kwargs={}), HumanMessagePromptTemplate(prompt=PromptTemplate(input_variables=['topic'], input_types={}, partial_variables={}, template='{topic}'), additional_kwargs={})]) middle=[ChatGroq(metadata={'lc_versions': {'langchain-core': '1.6.4', 'langchain': '1.4.2'}}, profile={'name': 'GPT OSS 20B', 'release_date': '2025-08-05', 'last_updated': '2026-05-27', 'open_weights': True, 'max_input_tokens': 131072, 'max_output_tokens': 65536, 'text_inputs': True, 'image_inputs': False, 'audio_inputs': False, 'video_inputs': False, 'text_outputs': True, 'image_outputs': False, 'audio_outputs': False, 'video_outputs':

In [9]:
draft_prompt = ChatPromptTemplate.from_messages([
    ("system", "You are a writer. Expand the given outline into a well-structured article of 150-200 words."),
    ("human", "{outline}"),
])

draft_chain = draft_prompt | get_llm(0.7) | StrOutputParser()

In [28]:
outline = outline_chain.invoke({"topic": "System design"})
draft = draft_chain.invoke({"outline": outline})

print(draft)
# Here problem was our topic will be lost from the chain beacuse draft only gets outline

**System Design – A Practical Roadmap**

Designing a robust system begins with a clear problem definition and requirement gathering. Capture functional goals—throughput, latency, cost—and non‑functional ones such as scalability and availability. User stories, edge cases, and failure scenarios help surface hidden constraints.

Next, sketch a high‑level architecture. Lay out the client, API gateway, micro‑services, data store, cache, messaging layer, and monitoring stack. Define interfaces and data flows to ensure each component knows what it consumes and produces.

Data modeling follows. Choose SQL, NoSQL, or NewSQL based on access patterns, and plan sharding, replication, backups, and eventual consistency. The persistence strategy should align with the system’s latency and durability needs.

Scalability, reliability, and performance tactics are then layered on. Employ load balancers, auto‑scaling, partitioning, caching, and CDNs. Plan disaster recovery, failover paths, graceful degrada

In [12]:
def add_outline(data: dict) -> dict:
    outline = outline_chain.invoke({"topic": data["topic"]})
    return {"topic": data["topic"], "outline": outline}

print(add_outline({"topic" : "Explain me about twiter"}))

{'topic': 'Explain me about twiter', 'outline': '**Twitter (now X)** – Quick‑fire outline\n\n- **Core identity**  \n  - Public microblogging platform where users post 280‑character “tweets” (text, images, videos, polls, threads).  \n  - Real‑time, public conversation hub; anyone can view content without an account, though interaction requires one.\n\n- **Key features**  \n  - **Timeline & Moments**: Personal feed of followed accounts, algorithm‑driven “For You” section, and curated “Moments” for trending topics.  \n  - **Interaction tools**: Reply, retweet, like, quote‑tweet, and “mute/block” to shape the experience.  \n  - **Multimedia & live**: Attach images, GIFs, threads, and stream live video via Periscope/Space.\n\n- **Business & monetization**  \n  - Revenue from advertising (promoted tweets, accounts, trends) and subscription tiers (Twitter Blue, X\u202f+).  \n  - Data licensing to news agencies and analytics firms.\n\n- **Cultural & political impact**  \n  - Platform for break

In [13]:
def add_draft(data: dict) -> dict:
    draft = draft_chain.invoke({"outline": data["outline"]})
    return {**data, "draft": draft}

In [14]:
pipeline_step1 = RunnableLambda(add_outline) | RunnableLambda(add_draft)

In [15]:
result = pipeline_step1.invoke({"topic": "Why system design matters for backend engineers"})

print("OUTLINE:\n", result["outline"])
print("\nDRAFT:\n", result["draft"])

OUTLINE:
 **Why System Design Matters for Backend Engineers**

- **Scalability & Performance**  
  • Enables architects to build services that grow with traffic while maintaining low latency and high throughput.

- **Reliability & Resilience**  
  • Provides blueprints for fault‑tolerance, graceful degradation, and automated recovery, reducing downtime.

- **Maintainability & Evolution**  
  • Encourages modular, loosely‑coupled components that can be updated or replaced without breaking the whole system.

- **Cross‑Team Collaboration**  
  • Acts as a shared language between backend, frontend, ops, and product teams, ensuring everyone understands trade‑offs and constraints.

- **Cost & Resource Optimization**  
  • Guides decisions on infrastructure, caching, data storage, and compute usage, helping teams stay within budget while meeting SLAs.

DRAFT:
 **Why System Design Matters for Backend Engineers**

In the world of high‑traffic applications, the architecture you choose determines

In [16]:
tweet_prompt = ChatPromptTemplate.from_messages([
    ("system", "Turn the given article into a punchy tweet, under 280 characters. Output ONLY the tweet."),
    ("human", "{draft}"),
])

linkedin_prompt = ChatPromptTemplate.from_messages([
    ("system", "Turn the given article into a professional LinkedIn post (3-4 short paragraphs). Output ONLY the post."),
    ("human", "{draft}"),
])

summary_prompt = ChatPromptTemplate.from_messages([
    ("system", "Summarize the given article in exactly 2 sentences. Output ONLY the summary."),
    ("human", "{draft}"),
])

tweet_chain = tweet_prompt | get_llm(0.7) | StrOutputParser()
linkedin_chain = linkedin_prompt | get_llm(0.7) | StrOutputParser()
summary_chain = summary_prompt | get_llm(0.3) | StrOutputParser()

In [17]:
parallel_outputs = RunnableParallel(
    tweet=tweet_chain,
    linkedin_post=linkedin_chain,
    summary=summary_chain,
)

In [23]:
full_pipeline = pipeline_step1 | RunnableLambda(lambda data: {"draft": data["draft"]}) | parallel_outputs

output = full_pipeline.invoke({"topic": "Why system design matters for backend engineers"})
print("TWEET:\n", output["tweet"])
print("\nLINKEDIN:\n", output["linkedin_post"])
print("\nSUMMARY:\n", output["summary"])

TWEET:
 System design isn’t just code—it’s the backbone that scales, stays reliable, and grows with your team. Good architecture handles traffic spikes, keeps data consistent, and makes future changes painless. Master it to boost your career to senior/architect level.

LINKEDIN:
 Building the backbone for millions of users isn’t just about writing code—it’s about crafting a resilient foundation that can grow with your product. Every design choice you make directly influences scalability, reliability, and future expansion.

**Scalability & Performance** – A well‑thought‑out architecture determines how gracefully an app handles traffic spikes, data volume, and latency. Selecting the right sharding strategy, caching layer, or load‑balancing technique keeps response times low even as usage explodes.

**Reliability & Maintainability** – Robust design ensures graceful degradation, data consistency, and rapid recovery. Redundant services, circuit breakers, and clear modular boundaries make ad

In [19]:
def is_long(data: dict) -> bool:
    return len(data["draft"].split()) > 100

short_path = RunnableLambda(lambda data: {"final": data["draft"], "note": "posted as-is (short)"})
long_path = RunnableLambda(lambda data: {"final": summary_chain.invoke({"draft": data["draft"]}), "note": "summarized (was long)"})

branch = RunnableBranch(
    (is_long, long_path),
    short_path,   # default, if no condition matches
)

In [20]:
branch_result = branch.invoke({"draft": result["draft"]})

print(branch_result["note"])
print(branch_result["final"])

summarized (was long)
Robust system design is essential for backend engineers because it establishes a strategic foundation that ensures scalability, reliability, maintainability, and cost efficiency for high‑traffic applications. By creating modular, resilient architectures and shared diagrams, teams can anticipate traffic spikes, reduce downtime, and align cross‑functional stakeholders, ultimately enabling continuous delivery and long‑term growth.


In [21]:
short_result = branch.invoke({"draft": "System design helps engineers build scalable systems."})

print(short_result["note"])
print(short_result["final"])

posted as-is (short)
System design helps engineers build scalable systems.
